In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from numpy import sqrt
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif

In [ ]:
datam = pd.read_excel("../../data1/CollegeDistance.xls")

In [ ]:
datam.head()

Оценим парную регрессию.

In [ ]:
model = sm.OLS.from_formula("ed ~ dist", data=datam).fit()
print(model.summary())

In [ ]:
model0 = sm.OLS.from_formula("ed ~ dist + black + hispanic", data=datam).fit()
print(model0.summary())

In [ ]:
model0.t_test(f"dist = {model.params['dist']}")

Ограничения вида $\mathbf{R}\mathbf{\beta} = \mathbf{r}$.

In [ ]:
R = np.zeros((1, 4))
R[0, 1] = 1
r = model.params["dist"]

model0.t_test((R, r))

In [ ]:
R = np.zeros((2, 4))
R[0, 1] = 1
R[1, 2] = 1
R[1, 3] = -1
r = [model.params["dist"], 0]

model0.t_test((R, r))

Оценим множественную регрессию.

In [ ]:
model1 = sm.OLS.from_formula(
    "ed ~ dist + bytest + female + black + hispanic + incomehi + ownhome + dadcoll + momcoll + cue80 + stwmfg80",
    data=datam,
).fit()
print(model1.summary())

In [ ]:
datam.corr()

Сравним новую оценку $\beta_{dist}$ со старой при помощи t-теста.

In [ ]:
model1.t_test(f"dist = {model.params['dist']}")

Рассчитаем $SER$ обоих регрессий.

In [ ]:
sqrt(model.ssr / model.df_resid)

In [ ]:
sqrt(model1.ssr / model1.df_resid)

In [ ]:
model1.f_test("black = 0, hispanic = 0")

In [ ]:
model1.f_test("black = hispanic = 0")

In [ ]:
R = np.zeros((2, 12))
R[0, 4] = 1
R[0, 5] = -1
R[1, 5] = 1

model1.f_test(R)

In [ ]:
model1.f_test("dist = -0.04, bytest = 0.1")

In [ ]:
R = np.zeros((2, 12))
R[0, 1] = 1
R[1, 2] = 1

r = [-0.04, 0.1]

model1.f_test((R, r))

In [ ]:
model1.f_test("dadcoll = 0, momcoll = 0")

В задании надо найти модельное время обучения в школе для Боба (см. задание). Создадим новый датафрейм с данными Боба.

In [ ]:
df_bob = pd.DataFrame(
    {
        "female": [0],
        "black": [1],
        "hispanic": [0],
        "dist": [2],
        "bytest": [58],
        "incomehi": [1],
        "ownhome": [1],
        "dadcoll": [0],
        "momcoll": [1],
        "cue80": [7.5],
        "stwmfg80": [9.75],
    }
)

In [ ]:
df_bob

In [ ]:
bob_ed = model1.predict(df_bob)
bob_ed

Джим имеет все характеристики такие же, как у Боба, кроме расстояния до школы (40 вместо 20).
Оценку числа лет обучения для Джима можно оценить как ${ED}_{Bob} + \beta_{dist} \times \Delta {ED}$.

In [ ]:
bob_ed + 2 * model1.params["dist"]

In [ ]:
df_jim = pd.DataFrame(
    {
        "female": [0],
        "black": [1],
        "hispanic": [0],
        "dist": [4],
        "bytest": [58],
        "incomehi": [1],
        "ownhome": [1],
        "dadcoll": [0],
        "momcoll": [1],
        "cue80": [7.5],
        "stwmfg80": [9.75],
    }
)

In [ ]:
model1.predict(df_jim)

Ожидаемо тот же результат.

# Мультиколлинеарность

In [ ]:
model = sm.OLS.from_formula(
    "ed ~ dist + bytest + female + black + hispanic + incomehi + ownhome + dadcoll + cue80 + stwmfg80",
    data=datam,
)
model_est = model.fit()
print(model_est.summary())

In [ ]:
for i in range(1, model.exog.shape[1]):
    vif_est = vif(model.exog, i)
    print(f"{model_est.params.index[i]:8}: {vif_est:5.4f}")

In [ ]:
datam["bytest_"] = datam.bytest + np.random.normal(size=3796, scale=2)

In [ ]:
datam[["bytest", "bytest_"]].corr()

In [ ]:
model3 = sm.OLS.from_formula(
    "ed ~ dist + bytest + bytest_ + female + black + hispanic + incomehi + ownhome + dadcoll + cue80 + stwmfg80",
    data=datam,
)
model3_est = model3.fit()
print(model3_est.summary())

In [ ]:
model3_est.f_test("bytest = 0, bytest_ = 0")

In [ ]:
for i in range(1, model3.exog.shape[1]):
    vif_est = vif(model3.exog, i)
    print(f"{model3_est.params.index[i]:8}: {vif_est:5.4f}")

In [ ]:
datam["male"] = 1 - datam.female

In [ ]:
model4 = sm.OLS.from_formula(
    "ed ~ dist + bytest + female + male + black + hispanic + incomehi + ownhome + dadcoll + cue80 + stwmfg80",
    data=datam,
)
model4_est = model4.fit()
print(model4_est.summary())

In [ ]:
for i in range(1, model4.exog.shape[1]):
    vif_est = vif(model4.exog, i)
    print(f"{model4_est.params.index[i]:8}: {vif_est:5.4f}")

In [ ]:
datam["bytest__"] = 1 + 30 * datam.bytest

In [ ]:
datam[["bytest", "bytest__"]].corr()

In [ ]:
model5 = sm.OLS.from_formula(
    "ed ~ dist + bytest + bytest__ + female + black + hispanic + incomehi + ownhome + dadcoll + cue80 + stwmfg80",
    data=datam,
)
model5_est = model5.fit()
print(model5_est.summary())

In [ ]:
for i in range(1, model5.exog.shape[1]):
    vif_est = vif(model5.exog, i)
    print(f"{model5_est.params.index[i]:8}: {vif_est:5.4f}")